**PSO: Particle Swarm Optimization**

A mathematical modelling algorithm, where the goal is to find the global minima of a fitness function.

**PSO with Classification Models**

To ensure a fair comparison against Encoders with genetic algorithm, the PSO will be trained on NBC model, then run by SVM and RF models.

In [5]:
import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

X_df = pd.read_excel('minmax.xlsx')
y_df = pd.read_csv('idC_with_header.csv')

X = X_df.to_numpy()
y = y_df.to_numpy()

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=X.shape[1], options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = X[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / X.shape[1]
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_selected = X[:, selected]

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)
nbc_acc = accuracy_score(y_test, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")

2025-04-19 17:12:31,732 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.0917
2025-04-19 17:20:06,958 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.09168464419475662, best pos: [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 

Best Cost:  0.09168464419475662 
Best POS:  [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 0
 1 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 0 0 1 1 1 0 1 1 1
 0 1 0 0 0 1 1 0 1 0 0 1 0 1 1 1 0 0 0 1 0 0 1 1 0 1 1 0 1 1 0 0 0 0 1 0 1
 0 0 1 0 1 0 1 1 0 0 1 0 1 1 0 1 0 0 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 0 0
 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 0 0 0 1 0 0 0 1 1 0 1 0 0

**Summary**

Accuracy results for models is slightly less than that of Encoders with GA implementation.

- SVM with PSO: 95%
- NBC with PSO: 87%
- RF with PSO: 93%

**Training with NBC**

**AE + PSO**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")
print(f"SVM Percision on features selected by PSO: {svm_percision * 100:.4f}")
print(f"SVM Recall on features selected by PSO: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on features selected by PSO: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on features selected by PSO: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on features selected by PSO: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on features selected by PSO: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by PSO: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by PSO: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by PSO: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by PSO: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2451
Epoch [2/50], Loss: 0.8251
Epoch [3/50], Loss: 0.6721
Epoch [4/50], Loss: 0.5663
Epoch [5/50], Loss: 0.5134
Epoch [6/50], Loss: 0.4967
Epoch [7/50], Loss: 0.4612
Epoch [8/50], Loss: 0.4411
Epoch [9/50], Loss: 0.4500
Epoch [10/50], Loss: 0.4648
Epoch [11/50], Loss: 0.4575
Epoch [12/50], Loss: 0.4648
Epoch [13/50], Loss: 0.3819
Epoch [14/50], Loss: 0.4137
Epoch [15/50], Loss: 0.3983
Epoch [16/50], Loss: 0.3923
Epoch [17/50], Loss: 0.3790
Epoch [18/50], Loss: 0.4172
Epoch [19/50], Loss: 0.3698
Epoch [20/50], Loss: 0.3855
Epoch [21/50], Loss: 0.4007
Epoch [22/50], Loss: 0.4186
Epoch [23/50], Loss: 0.4058
Epoch [24/50], Loss: 0.3538
Epoch [25/50], Loss: 0.3644
Epoch [26/50], Loss: 0.3863
Epoch [27/50], Loss: 0.4340
Epoch [28/50], Loss: 0.3715
Epoch [29/50], Loss: 0.4363
Epoch [30/50], Loss: 0.3950
Epoch [31/50], Loss: 0.3636
Epoch [32/50], Loss: 0.3913
Epoch [33/50], Loss: 0.3642
Epoch [34/50], Loss: 0.3873
Epoch [35/50], Loss: 0.3511


2025-05-03 10:11:35,356 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00156
2025-05-03 10:12:37,340 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0015625000000000014, best pos: [0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0]


Best Cost:  0.0015625000000000014 
Best POS:  [0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
Selected features:  [ 9 13 15 17 20]
SVM Accuracy on features selected by PSO: 91.0112
SVM Percision on features selected by PSO: 89.5543
SVM Recall on features selected by PSO: 90.0336
SVM F1-Score on features selected by PSO: 88.8878
Naïve Bayes Accuracy on features selected by PSO: 91.0112
Naïve Bayes Percision on features selected by PSO: 92.5458
Naïve Bayes Recall on features selected by PSO: 90.5678
Naïve Bayes F1-Score on features selected by PSO: 90.0224
Random Forest Accuracy on latent features selected by PSO: 83.1461
Random Forest Percision on latent features selected by PSO: 85.5886
Random Forest Recall on latent features selected by PSO: 84.2216
Random Forest F1-Score on latent features selected by PSO: 83.0564


**VAE + PSO**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()


import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")
print(f"SVM Percision on features selected by PSO: {svm_percision * 100:.4f}")
print(f"SVM Recall on features selected by PSO: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on features selected by PSO: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on features selected by PSO: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on features selected by PSO: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on features selected by PSO: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by PSO: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by PSO: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by PSO: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by PSO: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.5023
Epoch [2/50], Loss: 1.2950
Epoch [3/50], Loss: 1.1545
Epoch [4/50], Loss: 1.0269
Epoch [5/50], Loss: 0.9789
Epoch [6/50], Loss: 0.9565
Epoch [7/50], Loss: 0.8973
Epoch [8/50], Loss: 0.9037
Epoch [9/50], Loss: 0.7716
Epoch [10/50], Loss: 0.8267
Epoch [11/50], Loss: 0.7857
Epoch [12/50], Loss: 0.7745
Epoch [13/50], Loss: 0.7802
Epoch [14/50], Loss: 0.7438
Epoch [15/50], Loss: 0.7684
Epoch [16/50], Loss: 0.7364
Epoch [17/50], Loss: 0.7293
Epoch [18/50], Loss: 0.7604
Epoch [19/50], Loss: 0.7305
Epoch [20/50], Loss: 0.6948
Epoch [21/50], Loss: 0.6936
Epoch [22/50], Loss: 0.6561
Epoch [23/50], Loss: 0.6750
Epoch [24/50], Loss: 0.6669
Epoch [25/50], Loss: 0.6398
Epoch [26/50], Loss: 0.6764
Epoch [27/50], Loss: 0.6856
Epoch [28/50], Loss: 0.7049
Epoch [29/50], Loss: 0.6708
Epoch [30/50], Loss: 0.6876
Epoch [31/50], Loss: 0.6941
Epoch [32/50], Loss: 0.6449
Epoch [33/50], Loss: 0.6989
Epoch [34/50], Loss: 0.7290
Epoch [35/50], Loss: 0.6945


2025-05-03 10:17:07,561 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}


Epoch [50/50], Loss: 0.7017


pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00125
2025-05-03 10:18:07,407 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0012500000000000011, best pos: [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0]


Best Cost:  0.0012500000000000011 
Best POS:  [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0]
Selected features:  [12 16 19 22]
SVM Accuracy on features selected by PSO: 91.0112
SVM Percision on features selected by PSO: 91.0806
SVM Recall on features selected by PSO: 92.1398
SVM F1-Score on features selected by PSO: 89.9269
Naïve Bayes Accuracy on features selected by PSO: 91.0112
Naïve Bayes Percision on features selected by PSO: 91.2088
Naïve Bayes Recall on features selected by PSO: 92.8301
Naïve Bayes F1-Score on features selected by PSO: 91.0624
Random Forest Accuracy on latent features selected by PSO: 86.5169
Random Forest Percision on latent features selected by PSO: 89.0110
Random Forest Recall on latent features selected by PSO: 90.2660
Random Forest F1-Score on latent features selected by PSO: 87.3654


**Training with SVM**

**AE + PSO**

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(SVC(random_state=42),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")
print(f"SVM Percision on features selected by PSO: {svm_percision * 100:.4f}")
print(f"SVM Recall on features selected by PSO: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on features selected by PSO: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on features selected by PSO: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on features selected by PSO: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on features selected by PSO: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by PSO: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by PSO: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by PSO: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by PSO: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1371
Epoch [2/50], Loss: 0.7619
Epoch [3/50], Loss: 0.6555
Epoch [4/50], Loss: 0.6010
Epoch [5/50], Loss: 0.5983
Epoch [6/50], Loss: 0.5101
Epoch [7/50], Loss: 0.5224
Epoch [8/50], Loss: 0.4202
Epoch [9/50], Loss: 0.4116
Epoch [10/50], Loss: 0.4377
Epoch [11/50], Loss: 0.3898
Epoch [12/50], Loss: 0.4018
Epoch [13/50], Loss: 0.3592
Epoch [14/50], Loss: 0.4354
Epoch [15/50], Loss: 0.3771
Epoch [16/50], Loss: 0.4000
Epoch [17/50], Loss: 0.4004
Epoch [18/50], Loss: 0.3943
Epoch [19/50], Loss: 0.3883
Epoch [20/50], Loss: 0.3678
Epoch [21/50], Loss: 0.3893
Epoch [22/50], Loss: 0.4187
Epoch [23/50], Loss: 0.4115
Epoch [24/50], Loss: 0.3962
Epoch [25/50], Loss: 0.4233
Epoch [26/50], Loss: 0.3487
Epoch [27/50], Loss: 0.4552
Epoch [28/50], Loss: 0.4408
Epoch [29/50], Loss: 0.4175
Epoch [30/50], Loss: 0.4025
Epoch [31/50], Loss: 0.4072
Epoch [32/50], Loss: 0.3984
Epoch [33/50], Loss: 0.3839
Epoch [34/50], Loss: 0.4171
Epoch [35/50], Loss: 0.4151


2025-05-03 10:23:32,795 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}


Epoch [50/50], Loss: 0.3930


pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00156
2025-05-03 10:24:56,714 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0015625000000000014, best pos: [0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1]


Best Cost:  0.0015625000000000014 
Best POS:  [0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1]
Selected features:  [ 2 12 19 28 31]
SVM Accuracy on features selected by PSO: 91.0112
SVM Percision on features selected by PSO: 93.2692
SVM Recall on features selected by PSO: 93.2265
SVM F1-Score on features selected by PSO: 92.4700
Naïve Bayes Accuracy on features selected by PSO: 89.8876
Naïve Bayes Percision on features selected by PSO: 93.0633
Naïve Bayes Recall on features selected by PSO: 93.3761
Naïve Bayes F1-Score on features selected by PSO: 92.1201
Random Forest Accuracy on latent features selected by PSO: 89.8876
Random Forest Percision on latent features selected by PSO: 89.5202
Random Forest Recall on latent features selected by PSO: 92.5214
Random Forest F1-Score on latent features selected by PSO: 89.3556


**VAE + PSO**

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()


import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(SVC(random_state=42),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")
print(f"SVM Percision on features selected by PSO: {svm_percision * 100:.4f}")
print(f"SVM Recall on features selected by PSO: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on features selected by PSO: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on features selected by PSO: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on features selected by PSO: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on features selected by PSO: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by PSO: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by PSO: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by PSO: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by PSO: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.5408
Epoch [2/50], Loss: 1.2990
Epoch [3/50], Loss: 1.1743
Epoch [4/50], Loss: 1.0712
Epoch [5/50], Loss: 1.0403
Epoch [6/50], Loss: 0.9611
Epoch [7/50], Loss: 0.9297
Epoch [8/50], Loss: 0.8839
Epoch [9/50], Loss: 0.8346
Epoch [10/50], Loss: 0.8398
Epoch [11/50], Loss: 0.7561
Epoch [12/50], Loss: 0.7890
Epoch [13/50], Loss: 0.7946
Epoch [14/50], Loss: 0.8170
Epoch [15/50], Loss: 0.7504
Epoch [16/50], Loss: 0.7605
Epoch [17/50], Loss: 0.7348
Epoch [18/50], Loss: 0.7055
Epoch [19/50], Loss: 0.7203
Epoch [20/50], Loss: 0.6972
Epoch [21/50], Loss: 0.6963
Epoch [22/50], Loss: 0.7210
Epoch [23/50], Loss: 0.6838
Epoch [24/50], Loss: 0.6940
Epoch [25/50], Loss: 0.7505
Epoch [26/50], Loss: 0.6936
Epoch [27/50], Loss: 0.7139
Epoch [28/50], Loss: 0.6789
Epoch [29/50], Loss: 0.6956
Epoch [30/50], Loss: 0.6600
Epoch [31/50], Loss: 0.6952
Epoch [32/50], Loss: 0.6776
Epoch [33/50], Loss: 0.7329
Epoch [34/50], Loss: 0.7115
Epoch [35/50], Loss: 0.7183


2025-05-03 10:26:59,451 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}


Epoch [50/50], Loss: 0.6592


pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00125
2025-05-03 10:28:28,681 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0012500000000000011, best pos: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0]


Best Cost:  0.0012500000000000011 
Best POS:  [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0]
Selected features:  [ 0 15 21 29]
SVM Accuracy on features selected by PSO: 92.1348
SVM Percision on features selected by PSO: 88.2361
SVM Recall on features selected by PSO: 93.7612
SVM F1-Score on features selected by PSO: 88.9485
Naïve Bayes Accuracy on features selected by PSO: 94.3820
Naïve Bayes Percision on features selected by PSO: 94.9669
Naïve Bayes Recall on features selected by PSO: 95.9590
Naïve Bayes F1-Score on features selected by PSO: 95.1023
Random Forest Accuracy on latent features selected by PSO: 87.6404
Random Forest Percision on latent features selected by PSO: 85.8242
Random Forest Recall on latent features selected by PSO: 90.0371
Random Forest F1-Score on latent features selected by PSO: 84.6101
